# Bài 2 (Trang 28) - Dữ liệu Bank Marketing
## Câu 7: Biến đổi các thuộc tính (nhị phân, thứ tự, one-hot, nhóm tuổi, nhóm balance)

**Sinh viên:** Trần Đình Võ - **MSSV:** 6451071088

In [1]:
HO_TEN = "Trần Đình Võ"
MSSV = "6451071088"
print(f"Họ tên: {HO_TEN} - MSSV: {MSSV}")

Họ tên: Trần Đình Võ - MSSV: 6451071088


### Import thư viện và đọc dữ liệu

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

df = pd.read_csv("../Dataset/bank/bank.csv", sep=";")

### Các bước xử lý của câu 1 → 6 (chuẩn bị dữ liệu)

In [3]:
# Trong bộ dữ liệu, giá trị thiếu được ghi là 'unknown' -> chuyển về NaN
df = df.replace("unknown", np.nan)

df["subscribed"] = (df["y"] == "yes").astype(int)

df["contacts"] = df["campaign"] + df["previous"]          # tổng số lần liên hệ
df["contacted_before"] = (df["previous"] > 0).astype(int)  # đã từng được liên hệ ở chiến dịch trước

df = df.drop(columns=["poutcome", "pdays", "day"])

# job, education: điền bằng mode; contact: xem giá trị thiếu là một loại mới 'other'
df["job"] = df["job"].fillna(df["job"].mode()[0])
df["education"] = df["education"].fillna(df["education"].mode()[0])
df["contact"] = df["contact"].fillna("other")

### Câu 7: Biến đổi các thuộc tính (nhị phân, thứ tự, one-hot, nhóm tuổi, nhóm balance)

In [4]:
# Thuộc tính nhị phân yes/no -> 1/0
for col in ["default", "housing", "loan", "y"]:
    df[col] = df[col].map({"no": 0, "yes": 1})
df = df.drop(columns=["subscribed"])
# Thuộc tính thứ tự
df["education"] = df["education"].map({"primary": 1, "secondary": 2, "tertiary": 3})
months = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
df["month"] = df["month"].map({m: i + 1 for i, m in enumerate(months)})
# duration (giây) -> phút (số nguyên)
df["duration_min"] = (df["duration"] / 60).astype(int)

In [5]:
# Thuộc tính phân loại (nominal) -> One-hot encoding (tránh quan hệ thứ bậc giả)
df = pd.get_dummies(df, columns=["job", "marital", "contact"], dtype=int)

In [6]:
# Tuổi -> 8 nhóm có số lượng cân bằng
df["age_group"] = pd.qcut(df["age"], 8, labels=False)
# balance -> 6 nhóm có số lượng cân bằng
df["balance_group"] = pd.qcut(df["balance"], 6, labels=False)

In [7]:
print(pd.qcut(df["age"], 8).value_counts().sort_index())
print(pd.qcut(df["balance"], 6).value_counts().sort_index())

age
(18.999, 30.0]    632
(30.0, 33.0]      609
(33.0, 36.0]      599
(36.0, 39.0]      450
(39.0, 44.0]      638
(44.0, 49.0]      565
(49.0, 55.0]      533
(55.0, 87.0]      495
Name: count, dtype: int64
balance
(-3313.001, 2.0]     762
(2.0, 174.0]         750
(174.0, 444.0]       749
(444.0, 979.333]     753
(979.333, 2475.0]    753
(2475.0, 71188.0]    754
Name: count, dtype: int64


In [8]:
df.head(10)

,age,education,default,balance,housing,loan,month,duration,campaign,previous,...,job_technician,job_unemployed,marital_divorced,marital_married,marital_single,contact_cellular,contact_other,contact_telephone,age_group,balance_group
0,30,1,0,1787,0,0,10,79,1,0,...,0,1,0,1,0,1,0,0,0,4
1,33,2,0,4789,1,1,5,220,1,4,...,0,0,0,1,0,1,0,0,1,5
2,35,3,0,1350,1,0,4,185,1,1,...,0,0,0,0,1,1,0,0,2,4
3,30,3,0,1476,1,1,6,199,4,0,...,0,0,0,1,0,0,1,0,0,4
4,59,2,0,0,1,0,5,226,1,0,...,0,0,0,1,0,0,1,0,7,0
5,35,3,0,747,0,0,2,141,2,3,...,0,0,0,0,1,1,0,0,2,3
6,36,3,0,307,1,0,5,341,1,2,...,0,0,0,1,0,1,0,0,2,2
7,39,2,0,147,1,0,5,151,2,0,...,1,0,0,1,0,1,0,0,3,1
8,41,3,0,221,1,0,5,57,2,0,...,0,0,0,1,0,0,1,0,4,2
9,43,1,0,-88,1,1,4,313,1,2,...,0,0,0,1,0,1,0,0,4,0


### Nhận xét
- default, housing, loan, y chuyển về 0/1; education, month mã hóa theo thứ tự; job, marital, contact dùng One-hot encoding để tránh quan hệ thứ bậc giả.
- age chia 8 nhóm, balance chia 6 nhóm bằng qcut nên số phần tử mỗi nhóm cân bằng.